# VLA Manipulation Pipeline - Google Colab Verification & Execution Notebook

This notebook runs Phase 0 verification, task simulation, demo collection, SmolVLA fine-tuning, and evaluation.

## 1. Confirm GPU + CUDA

In [ ]:
import torch
print("Torch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA is not available. Please go to Runtime -> Change runtime type -> T4/A100 GPU.")

## 2. Install & Verify LeRobot CLI (with optional dataset/av dependencies)

In [ ]:
!pip install -q "lerobot[dataset]" av
!lerobot-train --help

## 3. Install & Verify ManiSkill3 (Headless EGL Rendering)

In [ ]:
# Install ManiSkill3 dependencies
!pip install -q --no-deps mani_skill
!pip install -q gymnasium h5py trimesh transforms3d pandas sapien dacite GitPython tyro

import os
import mani_skill.envs
import gymnasium as gym
import matplotlib.pyplot as plt

# Ensure headless EGL rendering configuration for Colab
os.environ["SAPIEN_RENDER_ENGINE"] = "EGL"

# Instantiate environment
env = gym.make("PickCube-v1", obs_mode="rgbd", render_mode="rgb_array")
obs, _ = env.reset()

# Step environment with random action
action = env.action_space.sample()
obs, reward, terminated, truncated, info = env.step(action)

# Render frame to verify visual output
frame = env.render()
if isinstance(frame, list):
    frame = frame[0]

plt.figure(figsize=(6, 6))
plt.imshow(frame)
plt.title("ManiSkill3 PickCube-v1 Verification Frame")
plt.axis("off")
plt.savefig("maniskill_verification_frame.png")
print("Frame successfully rendered and saved to maniskill_verification_frame.png")
env.close()

## 4. Phase 1: Scripted Baseline Policy Verification & 20-Trial Benchmark

Runs `ScriptedPickPlacePolicy` on `PickCube-v1` (pd_ee_delta_pose control mode) over 20 non-randomized trials.

In [ ]:
import numpy as np
import gymnasium as gym
import mani_skill.envs

class ScriptedPickPlacePolicy:
    def __init__(self, env):
        self.env = env
        self.reset()
        
    def reset(self):
        self.stage = 0  # 0: approach, 1: descend, 2: grasp, 3: lift, 4: move_to_goal, 5: release
        self.step_count = 0
        
    def get_action(self, obs):
        self.step_count += 1
        tcp_pose = obs["extra"]["tcp_pose"]
        obj_pose = obs["extra"]["obj_pose"]
        goal_pos = obs["extra"]["goal_pos"] if "goal_pos" in obs["extra"] else obj_pose[:3] + np.array([0, 0, 0.2])
        
        if hasattr(tcp_pose, "cpu"):
            tcp_pos = tcp_pose[0].cpu().numpy()[:3]
            obj_pos = obj_pose[0].cpu().numpy()[:3]
            goal_pos = goal_pos[0].cpu().numpy()[:3] if hasattr(goal_pos, "cpu") else goal_pos[:3]
        else:
            tcp_pos = tcp_pose[:3]
            obj_pos = obj_pose[:3]
            
        delta_pos = np.zeros(3)
        gripper_action = -1.0
        
        if self.stage == 0:
            target = obj_pos + np.array([0.0, 0.0, 0.10])
            diff = target - tcp_pos
            if np.linalg.norm(diff) < 0.02 or self.step_count > 30:
                self.stage = 1
                self.step_count = 0
            else:
                delta_pos = diff * 5.0
        elif self.stage == 1:
            target = obj_pos + np.array([0.0, 0.0, 0.015])
            diff = target - tcp_pos
            if np.linalg.norm(diff) < 0.015 or self.step_count > 25:
                self.stage = 2
                self.step_count = 0
            else:
                delta_pos = diff * 5.0
        elif self.stage == 2:
            target = obj_pos + np.array([0.0, 0.0, 0.015])
            delta_pos = (target - tcp_pos) * 2.0
            gripper_action = 1.0
            if self.step_count > 15:
                self.stage = 3
                self.step_count = 0
        elif self.stage == 3:
            target = obj_pos + np.array([0.0, 0.0, 0.25])
            diff = target - tcp_pos
            gripper_action = 1.0
            if (np.linalg.norm(diff[:2]) < 0.03 and tcp_pos[2] > 0.20) or self.step_count > 35:
                self.stage = 4
                self.step_count = 0
            else:
                delta_pos = diff * 4.0
        elif self.stage == 4:
            target = goal_pos
            diff = target - tcp_pos
            gripper_action = 1.0
            if np.linalg.norm(diff) < 0.03 or self.step_count > 40:
                self.stage = 5
                self.step_count = 0
            else:
                delta_pos = diff * 4.0
        elif self.stage == 5:
            delta_pos = np.zeros(3)
            gripper_action = -1.0
            
        delta_pos = np.clip(delta_pos, -1.0, 1.0)
        return np.array([delta_pos[0], delta_pos[1], delta_pos[2], 0.0, 0.0, 0.0, gripper_action], dtype=np.float32)

env = gym.make("PickCube-v1", obs_mode="state_dict", control_mode="pd_ee_delta_pose")
policy = ScriptedPickPlacePolicy(env)

num_trials = 20
successes = 0
grasps = 0

print("Running 20-Trial End-to-End Evaluation on PickCube-v1...")
for episode in range(num_trials):
    obs, info = env.reset(seed=1000 + episode)
    policy.reset()
    episode_success = False
    made_grasp = False
    
    for step in range(120):
        action = policy.get_action(obs)
        obs, reward, terminated, truncated, info = env.step(action)
        
        if info.get("is_grasped", False):
            made_grasp = True
        if info.get("success", False):
            episode_success = True
            break
            
    if made_grasp:
        grasps += 1
    if episode_success:
        successes += 1
        
    print(f"Trial {episode+1:02d}/{num_trials:02d}: Success={episode_success}, Grasped={made_grasp}")

env.close()
print(f"\nFINAL BASELINE BENCHMARK:")
print(f"Success Rate: {successes/num_trials * 100:.1f}% ({successes}/{num_trials})")
print(f"Grasp Success Rate: {grasps/num_trials * 100:.1f}% ({grasps}/{num_trials})")

## 5. Mounting Google Drive for Persistence

In [ ]:
from google.colab import drive
import os

# Mount Google Drive for persistent checkpoints and results storage
drive.mount('/content/drive')

project_dir = "/content/drive/MyDrive/vla-manipulation"
os.makedirs(project_dir, exist_ok=True)
print(f"Project persistence directory ready at: {project_dir}")